In [0]:
# Standard library imports
import os
import json
from functools import reduce
from collections import Counter
from itertools import chain

# PySpark core
from pyspark.sql import Window
from pyspark.sql import DataFrame
from pyspark.sql import functions as F

# PySpark functions
from pyspark.sql.functions import (
    col,
    lower,
    lit,
    create_map,
    explode,
    to_date,
    to_timestamp,
    sum,
    size,
    log1p,
    count,
    when,
    row_number, 
    desc
)

In [0]:
# Create a set of allowed regions
ALLOWED_REGIONS = {
    "CA", "DE", "FR", "GB", "IN",
    "JP", "KR", "MX", "RU", "US"
}

In [0]:
# Create the bronze ingestion function
def load_raw_csv(spark, path: str, region: str):
    
    # Validate region (fail fast)
    if region not in ALLOWED_REGIONS:
        raise ValueError(f"[INGESTION BLOCKED] Unknown region: {region}")

    # Read raw file
    df = (
    spark.read
    .option("header", True)
    .option("multiLine", True)
    .option("escape", "\"")
    .option("inferSchema", True)
    .csv(path)
    )

    # Add lineage column
    df = df.withColumn("region", lit(region))

    return df

In [0]:
def load_all_regions(spark, folder_path: str):
    
    dfs = []

    for file in os.listdir(folder_path):
        if file.endswith(".csv"):
            region = file.split("videos.csv")[0]
            print(f"Loading: {file} → region: {region}")  # ← add this

            df = load_raw_csv(
                spark,
                os.path.join(folder_path, file),
                region
            )

            dfs.append(df)

    print(f"Total DataFrames loaded: {len(dfs)}")  # ← and this

    if not dfs:
        raise ValueError(f"[INGESTION FAILED] No CSV files found in: {folder_path}")

    return reduce(lambda a, b: a.unionByName(b, allowMissingColumns=True), dfs)

In [0]:
# After union, verify no rows were lost
def validate_bronze(df, expected_regions: set):

    print("=== BASIC VALIDATION ===")

    required_cols = ["video_id", "views", "likes", "region"]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    if "_corrupt_record" in df.columns:
        bad = df.filter(col("_corrupt_record").isNotNull()).count()
        print(f"Corrupt records: {bad}")

    print("Row count:", df.count())

    # Reconciliation: confirm all expected regions loaded
    loaded_regions = {r.region for r in df.select("region").distinct().collect()}
    missing_regions = expected_regions - loaded_regions
    if missing_regions:
        print(f"[WARNING] Missing regions after union: {missing_regions}")
    else:
        print(f"[OK] All {len(loaded_regions)} regions present")

    df.select("region").distinct().show()

    return df

In [0]:
# Bronze to silver transformation: cleaning and casting
# Expects raw string columns from bronze layer: do not call twice.
def transform_silver(df):

    country_map = {
        "CA": "Canada", "DE": "Germany", "FR": "France",
        "GB": "Great Britain", "IN": "India", "JP": "Japan",
        "KR": "South Korea", "MX": "Mexico", "RU": "Russia",
        "US": "United States"
    }

    mapping_expr = create_map([lit(x) for x in chain(*country_map.items())])

    return (
        df
        .filter(col("video_id").isNotNull())
        .filter(~col("video_id").startswith("#"))
        .withColumn("publish_time", to_timestamp("publish_time"))
        .withColumn("trending_date", to_date("trending_date", "yy.dd.MM"))
        .withColumn("views", col("views").cast("long"))
        .withColumn("likes", col("likes").cast("long"))
        .withColumn("dislikes", col("dislikes").cast("long"))
        .withColumn("comment_count", col("comment_count").cast("long"))
        .withColumn("country", mapping_expr[col("region")])
        .select(
            "video_id",
            "category_id",
            "title",
            "channel_title",
            "publish_time",
            "trending_date",
            "views",
            "likes",
            "dislikes",
            "comment_count",
            "comments_disabled",
            "ratings_disabled",
            "video_error_or_removed",
            "region",
            "country"
        )
    )

In [0]:
# Deduplicate trending videos
def deduplicate_trending(df):
    """
    Each video can appear multiple times per region (one row per trending day).
    Keep the last trending snapshot per video per region — peak engagement stats.
    """

    window = Window.partitionBy("video_id", "region").orderBy(desc("trending_date"))

    df_deduped = (
        df
        .withColumn("rn", row_number().over(window))
        .filter(col("rn") == 1)
        .drop("rn")
    )

    before = df.count()
    after = df_deduped.count()
    print(f"Deduplication: {before:,} → {after:,} rows ({before - after:,} removed)")

    return df_deduped

In [0]:
def validate_silver(df_silver, df_bronze):

    print("=== SILVER VALIDATION ===")

    # Row counts
    bronze_count = df_bronze.count()
    silver_count = df_silver.count()
    removed = bronze_count - silver_count
    print(f"Bronze rows:  {bronze_count:,}")
    print(f"Silver rows:  {silver_count:,}")
    print(f"Rows removed: {removed:,} ({removed/bronze_count:.1%})")

    # No duplicate video+region combinations
    dupes = (
        df_silver
        .groupBy("video_id", "region")
        .agg(count("*").alias("n"))
        .filter(col("n") > 1)
        .count()
    )
    if dupes > 0:
        raise ValueError(f"[VALIDATION FAILED] {dupes:,} duplicate video+region combinations remain")
    else:
        print(f"[OK] No duplicate video+region combinations")

    # No negative values in numeric columns
    numeric_cols = ["views", "likes", "dislikes", "comment_count"]
    for c in numeric_cols:
        neg = df_silver.filter(col(c) < 0).count()
        if neg > 0:
            raise ValueError(f"[VALIDATION FAILED] {neg:,} negative values in {c}")
    print(f"[OK] No negative values in numeric columns")

    # Publish time before trending date
    late = df_silver.filter(col("publish_time") > col("trending_date"))
    late_count = late.count()

    if late_count > 0:
        # Check if same-day (publish date == trending date, just later in the day)
        same_day = late.filter(
            to_date(col("publish_time")) == col("trending_date")
        ).count()

        different_day = late_count - same_day

        if different_day > 0:
            raise ValueError(
            f"[VALIDATION FAILED] {different_day:,} rows where publish_time is genuinely after trending_date (different day) — investigate"
        )
        else:
            print(f"[WARNING] {same_day:,} rows where publish_time is after trending_date but same day — known timestamp quirk, not blocking")
    else:
        print(f"[OK] All publish times precede trending date")
    
    # All region codes mapped to known countries
    unknown_regions = df_silver.filter(col("country").isNull()).count()
    if unknown_regions > 0:
        raise ValueError(f"[VALIDATION FAILED] {unknown_regions:,} rows with unrecognized region code")
    else:
        print(f"[OK] All region codes mapped to country names")

    # All regions present
    regions = {r.region for r in df_silver.select("region").distinct().collect()}
    print(f"[OK] Regions present: {sorted(regions)}")

    print("=== SILVER VALIDATION PASSED ===")

    return df_silver

In [0]:
# Load a single category JSON file
def load_raw_json(spark, file_path: str, region: str):

    df = (
        spark.read
             .option("multiline", "true")
             .json(file_path)
    )

    if df.limit(1).count() == 0:
        raise ValueError(f"[INGESTION FAILED] File is empty or unreadable: {file_path}")

    df = df.withColumn("region", lit(region))

    return df

In [0]:
# Create a batch loader for all category JSONs
def load_all_category_jsons(spark, folder_path: str):

    dfs = {}

    for file in os.listdir(folder_path):
        if file.endswith(".json"):
            region = file.split("_category_id")[0]

            df = load_raw_json(
                spark,
                os.path.join(folder_path, file),
                region
            )

            dfs[region] = df

    if not dfs:
        raise ValueError(f"[INGESTION FAILED] No JSON files found in: {folder_path}")

    return dfs

In [0]:
# Create function to validate raw category JSONs
def validate_category_raw(dfs: dict) -> dict:
    """
    Schema sanity check across all region DataFrames.
    Logs mismatches and row counts, then returns the dfs dict.
    """

    if not dfs:
        raise ValueError("[VALIDATION FAILED] Empty dataframe dictionary passed.")

    # Use simpleString() for reliable schema comparison
    base_region, base_df = next(iter(dfs.items()))
    base_schema_str = base_df.schema.simpleString()

    valid_dfs = {}

    for region, df in dfs.items():
        schema_str = df.schema.simpleString()

        if schema_str != base_schema_str:
            print(f"[WARNING] Schema mismatch in {region} vs {base_region} — skipping")
            print(f"  Expected: {base_schema_str}")
            print(f"  Got:      {schema_str}")
            continue

        row_count = df.count()
        print(f"[OK] {region}: rows={row_count}")
        valid_dfs[region] = df

    if not valid_dfs:
        raise ValueError("[VALIDATION FAILED] No DataFrames passed schema validation.")

    return valid_dfs  

In [0]:
def flatten_categories(df):
    """Explode items array and select relevant fields."""

    return (
        df
        .select(explode("items").alias("item"), "region")
        .select(
            col("item.id").alias("category_id"),
            col("item.snippet.title").alias("category_name"),
            "region"
        )
    )

In [0]:
# Build category dimension df `build_category_dim` function
def build_category_dim(category_valid_dfs):
    category_unioned_df = reduce(
        DataFrame.unionByName,
        category_valid_dfs.values()
    )

    category_flattened_df = flatten_categories(category_unioned_df)

    category_dim_df = (
        category_flattened_df
        .dropDuplicates(["category_id"])
        .drop("region")
        .orderBy("category_id")
    )

    return category_dim_df

In [0]:
# Create a safe division function to avoid NULL entries
def safe_div(numerator, denominator):
    return when(denominator > 0, numerator / denominator).otherwise(lit(0.0))

In [0]:
# Creat a function to add some business metrics: engagement_rate, comment_rate, log_views
def add_business_metrics(df):

    return (
        df
        .withColumn(
            "engagement_rate",
            when(
                col("comments_disabled") == False,
                safe_div(col("likes") + col("dislikes") + col("comment_count"), col("views"))
            ).otherwise(
                safe_div(col("likes") + col("dislikes"), col("views"))
            )
        )
        .withColumn(
            "comment_rate",
            when(
                col("comments_disabled") == False,
                safe_div(col("comment_count"), col("views"))
            ).otherwise(None)
        )
        .withColumn(
            "like_ratio",
            when(
                col("ratings_disabled") == False,
                safe_div(col("likes"), col("views"))
            ).otherwise(None)
        )
        .withColumn(
            "log_views",
            log1p(col("views"))
        )
    )

In [0]:
def finalize_gold_schema(df):

    return df.select(
        "video_id",
        "title",
        "channel_title",
        "category_name",
        "publish_time",
        "trending_date",
        "views",
        "likes",
        "dislikes",
        "comment_count",
        "comments_disabled",
        "ratings_disabled",
        "video_error_or_removed",
        "region",
        "country",
        "like_ratio",
        "engagement_rate",
        "comment_rate",
        "log_views"
    )

In [0]:
# Validate df_gold, specifically new business metrics
def validate_gold(df_gold, df_silver):

    print("=== GOLD VALIDATION ===")

    # Row count should match silver
    silver_count = df_silver.count()
    gold_count = df_gold.count()

    if gold_count != silver_count:
        raise ValueError(f"[VALIDATION FAILED] Row count mismatch — silver: {silver_count:,}, gold: {gold_count:,}")
    else:
        print(f"[OK] Row count matches silver: {gold_count:,}")

    # Schema check — all expected columns present
    expected_cols = [
        "video_id", "title", "channel_title", "category_name",
        "publish_time", "trending_date", "views", "likes", "dislikes",
        "comment_count", "comments_disabled", "ratings_disabled",
        "video_error_or_removed", "region", "country", "like_ratio",
        "engagement_rate", "comment_rate", "log_views"
    ]
    missing = [c for c in expected_cols if c not in df_gold.columns]
    if missing:
        raise ValueError(f"[VALIDATION FAILED] Missing columns: {missing}")
    else:
        print(f"[OK] All expected columns present")

    # Engagement rate bounds — should be between 0 and 1
    out_of_bounds = df_gold.filter(
        (col("engagement_rate") < 0) | (col("engagement_rate") > 1)
    ).count()
    if out_of_bounds > 0:
        raise ValueError(f"[VALIDATION FAILED] {out_of_bounds:,} rows with engagement_rate outside 0-1")
    else:
        print(f"[OK] All engagement_rate values within bounds")

    # comment_rate null only where comments_disabled
    unexpected_null_comments = df_gold.filter(
        (col("comments_disabled") == False) & col("comment_rate").isNull()
    ).count()
    if unexpected_null_comments > 0:
        raise ValueError(f"[VALIDATION FAILED] {unexpected_null_comments:,} rows with null comment_rate where comments are enabled")
    else:
        print(f"[OK] comment_rate nulls align with comments_disabled flag")

    # like_ratio null only where ratings_disabled
    unexpected_null_ratings = df_gold.filter(
        (col("ratings_disabled") == False) & col("like_ratio").isNull()
    ).count()
    if unexpected_null_ratings > 0:
        raise ValueError(f"[VALIDATION FAILED] {unexpected_null_ratings:,} rows with null like_ratio where ratings are enabled")
    else:
        print(f"[OK] like_ratio nulls align with ratings_disabled flag")

    # log_views should always be positive
    bad_log = df_gold.filter(col("log_views") <= 0).count()
    if bad_log > 0:
        raise ValueError(f"[VALIDATION FAILED] {bad_log:,} rows with log_views <= 0")
    else:
        print(f"[OK] All log_views values positive")

    print("=== GOLD VALIDATION PASSED ===")

    return df_gold

In [0]:
# Write to table
def write_gold_table(df, table_name, mode="overwrite", overwrite_schema=True):

    (df
        .write
        .format("delta")
        .mode(mode)
        .option("overwriteSchema", str(overwrite_schema).lower())
        .saveAsTable(table_name))